# DP-OT on real data — OGB-arxiv temporal shift

**Purpose:** stress-test the synthetic result. On the Gaussian-mixture data the covariate shift *is* a clean reweighting of the same clusters the prototypes recover — the method's ideal case. This notebook asks the honest question: **does DP prototype-mass reweighting still recover an adaptation gain when the shift is real and not cluster-structured?**

Setup: OGB-arxiv citation graph, split temporally — papers **before 2018** are the public source, papers **from 2018 on** are the private target. Topic drift over time induces the distribution shift. ~170k nodes, 128-d features, 40 classes.

Honest expectation: the oracle gain here may be **small or even zero** — and that is itself the informative result. A small/zero gain means real shift isn't captured by prototype mass; a clear gain that the DP histogram recovers is strong evidence the method generalizes beyond its home turf.

**Use a GPU** (Runtime → Change runtime type → GPU). The graph is large; KMeans + GNN training are recomputed per run, so keep the sweep grid modest at first.

## 0. Clone + install

In [ ]:
import os

REPO_URL = "https://github.com/ChefAltoids/MM-edgeDP"
BRANCH   = "dp-ot"
REPO_DIR = "/content/MM-edgeDP"

if not os.path.isdir(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git fetch origin && git checkout $BRANCH && git pull --ff-only
os.chdir(REPO_DIR)

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
!pip install -q torch_geometric ogb
print("\nReady.")

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from dp_ot.run_experiment import run_experiment
from dp_ot.sweep import run_sweep
from dp_ot.eval import plots

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

def show_results(results: dict) -> pd.DataFrame:
    df = pd.DataFrame(results).T[["auroc", "acc", "f1", "proto_l1_error"]]
    order = ["source_only", "dp_histogram", "dp_exponential", "oracle", "target_oracle"]
    return df.reindex([m for m in order if m in df.index]).round(4)

def adaptation_diagnostic(results):
    so  = results['source_only']['auroc']
    orc = results['oracle']['auroc']
    tgt = results['target_oracle']['auroc']
    gap = orc - so
    print(f'source_only   AUROC : {so:.4f}')
    print(f'oracle        AUROC : {orc:.4f}  (gain over source: {gap:+.4f})')
    print(f'target_oracle AUROC : {tgt:.4f}  (upper bound)')
    if gap <= 1e-3:
        print('  -> no non-private oracle gain: real shift is not captured by prototype mass.')
        print('     This is a legitimate (negative) finding, not a bug.')
        return
    for m in ['dp_histogram', 'dp_exponential']:
        a  = results[m]['auroc']; l1 = results[m]['proto_l1_error']
        print(f'  {m:15s}: recovers {(a-so)/gap*100:6.1f}% of oracle gain  (AUROC {a:.4f}, proto_l1 {l1:.3f})')

## 1. Load and characterize the shift
Before running anything, quantify *how much* the source and target differ. If feature/label shift is negligible, there is nothing to adapt to and the rest is moot. (First run downloads ~80 MB.)

In [ ]:
from dp_ot.data.real_splits import load_ogb_arxiv_temporal, compute_feature_shift

G_source, G_target = load_ogb_arxiv_temporal(source_before_year=2018, target_from_year=2018, root="data/ogb")
print(f"source: {G_source.num_nodes:>7d} nodes, {G_source.edge_index.shape[1]:>8d} edges")
print(f"target: {G_target.num_nodes:>7d} nodes, {G_target.edge_index.shape[1]:>8d} edges")

fs = compute_feature_shift(G_source, G_target)
print(f"\nfeature mean-shift norm : {fs['mean_shift_norm']:.4f}")
print(f"feature linear MMD^2    : {fs['linear_mmd2']:.4f}")

ys, yt = G_source.y.numpy(), G_target.y.numpy()
C = int(max(ys.max(), yt.max())) + 1
hs = np.bincount(ys, minlength=C) / len(ys)
ht = np.bincount(yt, minlength=C) / len(yt)
print(f"\nlabel-distribution L1 shift (sum|p_s - p_t|): {np.abs(hs - ht).sum():.4f}  (0=identical, 2=disjoint)")
top = np.argsort(ht - hs)[::-1][:5]
print("classes that grew most in the target:", top.tolist())

## 2. Single run (ε = 1.0)
The full pipeline on real data. `target_oracle` (trained on the target) is the upper bound; `source_only` ignores the shift; `oracle` is the non-private reweighting; the two `dp_*` rows are the private estimates. Watch the diagnostic for whether there is a real oracle gain to recover.

In [ ]:
ogb_cfg = {
    "dataset": "ogb_arxiv",
    "ogb_source_before_year": 2018, "ogb_target_from_year": 2018, "ogb_root": "data/ogb",
    "K": 64, "d_max": 20, "B": 5.0,
    "epsilon": 1.0, "seed": 0,
    "hidden": 128, "num_layers": 2, "epochs": 150, "lr": 0.005, "weight_decay": 5e-4,
    "device": DEVICE,
}
results = run_experiment(ogb_cfg)
display(show_results(results))
adaptation_diagnostic(results)

## 3. Privacy sweep (ε)
Privacy-utility curve across seeds. **Start small** — each combination retrains the GNNs and recomputes KMeans on the full graph, so this is minutes per combination even on a GPU. Widen `epsilon`/`seed` once a single run looks worthwhile.

In [ ]:
sweep_cfg = {
    "dataset": "ogb_arxiv",
    "ogb_source_before_year": 2018, "ogb_target_from_year": 2018, "ogb_root": "data/ogb",
    "d_max": 20, "B": 5.0,
    "hidden": 128, "num_layers": 2, "epochs": 150, "lr": 0.005, "weight_decay": 5e-4,
    "device": DEVICE,
    "out": "dp_ot/outputs/sweep_ogb_arxiv.csv",
    "sweep": {
        "epsilon": [0.3, 1.0, 3.0, 10.0],
        "K":       [64],
        "seed":    [0, 1, 2],
    },
}
df = run_sweep(sweep_cfg)

# gamma is unused for real data (stays at the default 0.5 in every row), so plot at gamma=0.5.
plots.plot_auroc_vs_epsilon(df, gamma=0.5, out_path="dp_ot/outputs/ogb_auroc_vs_eps.pdf")
plots.plot_l1_vs_epsilon(df,   gamma=0.5, out_path="dp_ot/outputs/ogb_l1_vs_eps.pdf")
plt.show()
df.tail(10)

## 4. Save outputs

In [ ]:
import shutil
shutil.make_archive("/content/dp_ot_real_outputs", "zip", "dp_ot/outputs")
try:
    from google.colab import files
    files.download("/content/dp_ot_real_outputs.zip")
except Exception as e:
    print("Download skipped:", e)